In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, DataCollatorForSeq2Seq
from torch.utils.data import Dataset
import random
import os
import json
from tqdm import tqdm

In [96]:
def set_seed(seed=42):
    random.seed(seed)
#     np.random.seed(seed)
#     tf.random.set_seed(seed)
#     torch.manual_seed(seed)
#     torch.cuda.manual_seed(seed)
#     torch.cuda.manual_seed_all(seed)
#     torch.backends.cudnn.deterministic = True
#     torch.backends.cudnn.benchmark = False

set_seed(42)


In [97]:

'''
load model, tokenizer
'''

def load_model():
    model_name = "Qwen/Qwen2.5-0.5B"

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype="auto",
        device_map="auto"
    )
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print('using device:', device)

    model.to(device)
    return model, tokenizer


In [98]:
'''
data part
'''
data_dict = {"hotpot_train": "./dataset/hotpotQA/hotpot_train_v1.1.json",
               "hotpot_test": "./dataset/hotpotQA/hotpot_dev_distractor_v1.json",
               "squad_train": "./dataset/squad2.0/train-v2.0.json",
               "squad_test": "./dataset/squad2.0/dev-v2.0.json"
               }

MAX_SEQ_LEN = 5000
MAX_QES_LEN = 100
MAX_TRAIN_NUM = 10000
"""
数据格式
{"title":"", context":"", "question":"", "answer":"", answer_idx:[(start, end), (), ()]}
"""
import copy

def data_augmentation(data, corpus):
    print("doing augmentation now")
    
    cnt = 0
    new_data = []
    for example in tqdm(data):
        new_data.append(copy.deepcopy(example))
        context = copy.deepcopy(example["context"])
#         if random.random() < 0.3:
#             new_context = ""
#             for idx_pair in example["answer_idx"]:

#                 new_context += context[idx_pair[0]:idx_pair[1]]
#         else:
        new_context = ""
        supporting = []
        for idx_pair in example["answer_idx"]:
            replace_content = random.sample(corpus, 1)[0]
#                 print(type(replace_content))
            new_context += replace_content + context[idx_pair[0]:idx_pair[1]]
            supporting.append(context[idx_pair[0]:idx_pair[1]])
        
        if len(new_context) > MAX_SEQ_LEN - MAX_QES_LEN:
            continue
            
#         example["old_context"] = example["context"]
        example["context"] = new_context
        if cnt == 0:
            
            print("**************sample example**************")
            print("question:", example["question"])
            print("answer:", example["answer"])
#             print("supporting_fact:", supporting)
#             print("context:", context)
            print("before:", new_data[-1]["context"])
            print("after:", example["context"])
            cnt +=1
        new_data.append(example)
    return new_data

In [106]:
                
def hotpotQA_dataload(file_path, da = False):
    data_list = []
    max_len = 0
    corpus = []
    
    with open(file_path) as f:
        text = json.loads(f.read())
        for idx, item in tqdm(enumerate(text)):

            ans = item["answer"]
            question = item["question"]
            title = [t[0] for t in item["context"]]
            context = ["".join(t[1]) for t in item["context"]]
            corpus.extend(context)
            title2context = {}
            for _t, _c in zip(title, context):
                title2context[_t] = _c
            context = "".join(context)
            if len(context) > MAX_SEQ_LEN - MAX_QES_LEN or len(question) > MAX_QES_LEN:
                continue
            supporting_sentence = ["".join(title2context[t[0]]) for t in item["supporting_facts"]]

            answer_idx = []
            for sent in supporting_sentence:
                answer_start = context.find(sent)
                answer_idx.append((answer_start, answer_start + len(sent)))
            new_instance = {"title": ".".join(title), "context": context, "question": question, "answer": ans,
                            "answer_idx": answer_idx}
            #             if idx % 100 == 0:
            #                 print(new_instance)
            data_list.append(new_instance)
#             max_len = max(max_len, len(context + question))
    if da:
        data = data_augmentation(data_list, corpus)
        
    train_num = min(MAX_TRAIN_NUM*2 if da else MAX_TRAIN_NUM, len(data_list))
    data_list = random.sample(data_list, k=train_num)
    return data_list, corpus


def squad_dataload(file_path, da=False):
    data_list = []
    max_len = 0
    corpus = []
    
    with open(file_path) as f:
        text = json.loads(f.read())
        data = text["data"]
        for example in tqdm(data):
            title = example["title"]
            for each in example["paragraphs"]:
                context = each["context"]
                if len(context) > MAX_SEQ_LEN - MAX_QES_LEN:
                    continue
                #             print(context)
                for _v in each["qas"]:
                    try:
                        question = _v["question"]
                        if len(question) > MAX_QES_LEN:
                            continue
                        #                     print(_v["answers"])
                        answer = _v["answers"][0]["text"]
                        answer_start = _v["answers"][0]["answer_start"]
                        while answer_start >=0 and context[answer_start] not in [".", "?", "!"]:
                            answer_start -= 1
                        answer_start +=1
                        new_instance = {"title": title, "context": context, "question": question, "answer": answer,
                                        "answer_idx": [(answer_start, len(context))]}
                        #                 print(new_instance)
                        data_list.append(new_instance)
                        max_len = max(max_len, len(context + question))
                        corpus.append(context[:answer_start])

                    except Exception as e:
                        pass
    if da:
        data = data_augmentation(data_list, corpus)
        
    train_num = min(MAX_TRAIN_NUM*2 if da else MAX_TRAIN_NUM, len(data_list))
    data_list = random.sample(data_list, k=train_num)
    return data_list, corpus

class CustomDataset(Dataset):
    def __init__(self, name, tokenizer, post="train", da = False):
        self.data, _ = self.dataloader(data_dict[f"{name}_{post}"], da)
        self.tokenizer = tokenizer
        self.input_data = []

        self.end_token_ids = self.tokenizer.convert_tokens_to_ids('<|im_end|>')
        prompt1 = "<|im_start|>Please answer the question according to the given context.\n context:"
        self.prompt1_tokens = self.tokenizer(prompt1, add_special_tokens=False)

        prompt2 = "\nquestion:"
        self.prompt2_tokens = self.tokenizer(prompt2, add_special_tokens=False)

        #         print(self.tokenizer.tokenize(prompt2))
        #         print(self.prompt2_tokens)

        self.tokenize()
    

    def dataloader(self, file_path, da = False):
        write_path = file_path+".train_"+str(int(da))
        data, corpus = [], []
        if os.path.exists(write_path):
            print("loading from:{}".format(write_path))
            with open(write_path) as f:
                for line in tqdm(f):
                    data.append(json.loads(line.strip()))
                
        else:
            print("loading from:{}".format(file_path))

            if "squad" in file_path:
                data, corpus = squad_dataload(file_path, da)

            elif "hotpot" in file_path:
                data, corpus = hotpotQA_dataload(file_path, da)
            else:
                print("invalid data data, please select from (hotpotQA, squad2.0)")
                return
                
            with open(write_path, "w") as f:
                for item in data:
                    f.write(json.dumps(item)+"\n")
                print("data feature write in :", write_path)

        #     print(data[:10])
        if "train" in file_path:
            print("train_num:{}".format(len(data)))
        else:
            print("eval_num:{}".format(len(data)))
        return data, corpus
    
    def tokenize(self):
        print("begin tokenize")
        for item in tqdm(self.data):
            context_tokens = self.tokenizer(item["context"], add_special_tokens=False)
            question_tokens = self.tokenizer(item["question"], add_special_tokens=False)
            label_tokens = self.tokenizer(item["answer"], add_special_tokens=False)

            max_context_len = MAX_SEQ_LEN - len(self.prompt1_tokens["input_ids"]) - len(self.prompt2_tokens)
            max_question_len = MAX_QES_LEN - 1

            input_ids = (
                    self.prompt1_tokens["input_ids"] +
                    context_tokens["input_ids"][:max_context_len] +
                    self.prompt2_tokens["input_ids"] +
                    question_tokens["input_ids"][:max_question_len] +
                    [self.end_token_ids] +
                    label_tokens["input_ids"] +
                    [self.end_token_ids]
            )
            attention_mask = (
                    self.prompt1_tokens["attention_mask"] +
                    context_tokens["attention_mask"][:max_context_len] +
                    self.prompt2_tokens["attention_mask"] +
                    question_tokens["attention_mask"][:max_question_len] +
                    [1] +
                    label_tokens["attention_mask"] +
                    [1]
            )

            labels = (
                    [-100] * (len(input_ids) - len(label_tokens["input_ids"]) - 1)
                    + label_tokens["input_ids"]
                    + [self.end_token_ids]
            )

            self.input_data.append({"input_ids": input_ids,
                                    "attention_mask": attention_mask,
                                    "labels": labels, })
        print("tokenization done!")

    def __getitem__(self, idx):
        # 返回一个字典，包含 input_ids, attention_mask, 以及 labels (如果有的话)
        return self.input_data[idx]

    def __len__(self):
        return len(self.input_data)

In [100]:
# def dataloader(file_path, da=False):
#         data = []
#         print("loading from:{}".format(file_path))

#         if "squad" in file_path:
#             data, max_len = squad_dataload(file_path, da)
#         elif "hotpot" in file_path:
#             data, max_len = hotpotQA_dataload(file_path, da)
#         else:
#             print("invalid data data, please select from (hotpotQA, squad2.0)")

#         #     print(data[:10])
#         if "train" in file_path:
#             print("train_num:{}".format(len(data)))
#         else:
#             print("eval_num:{}".format(len(data)))
#         return data, max_len
    
# name = "squad"
# post = "train"
# data, max_len = dataloader(data_dict[f"{name}_{post}"], da=True) 

# name = "hotpot"
# post = "train"
# data, max_len = dataloader(data_dict[f"{name}_{post}"], da=True) 

In [101]:
from peft import LoraConfig, TaskType, get_peft_model
def lora_config():
# set lora
    config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        target_modules=[
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
        ],
        inference_mode=False,  # 训练模式
        r=8,  # Lora 秩
        lora_alpha=32,  # Lora alaph，具体作用参见 Lora 原理
        lora_dropout=0.1,  # Dropout 比例
    )
    return config

from swanlab.integration.huggingface import SwanLabCallback
import swanlab
def load_swanlab_callback():
    swanlab_callback = SwanLabCallback(
        project="Qwen2-finetune",
        experiment_name="Qwen2-0.5B",
        description="Qwen2-finetune",
    #     config={
    #         "model": "qwen/Qwen2-1.5B-Instruct",
    #         "dataset": "huangjintao/zh_cls_fudan-news",
    #     },
    )
    return swanlab_callback

In [111]:
# '''
# main
# '''
model, tokenizer = load_model()
train_dataset = CustomDataset("hotpot", tokenizer, post="train", da=True)
# train_dataset = CustomDataset("hotpot", tokenizer, post="train", da=True)
# eval_dataset = CustomDataset("hotpot", tokenizer, post="test")


using device: cuda
loading from:./dataset/hotpotQA/hotpot_train_v1.1.json.train_0


10000it [00:00, 37921.81it/s]


train_num:10000
begin tokenize


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10000/10000 [00:35<00:00, 281.00it/s]

tokenization done!


In [ ]:
swanlab_callback = load_swanlab_callback()
#set args

args = TrainingArguments(
    output_dir="./output/Qwen2",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=16,
    logging_steps=10,
    num_train_epochs=2,
    save_steps=100,
    learning_rate=1e-4,
    save_on_each_node=True,
    gradient_checkpointing=True,
    report_to="none",
    fp16=True,
)

config = lora_config()
model = get_peft_model(model, config)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset =eval_dataset，
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
    callbacks=[swanlab_callback],
)

trainer.train()

In [ ]:

# '''
# inference
# '''   
# import time
# prompt = "<|im_start|>How to train a CNN model\n<|im_end|>"
# messages = [
#     {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
#     {"role": "user", "content": prompt}
# ]
# messages = prompt
# # tensor([[151644,   8948,    198,    283,    525,   1207,  16948,     11,   3465,
# #             553,  54364,  14817,     13,   1446,    525,    264,  10950,  17847,
# #              13, 151645,    198, 151644,    872,    198,   4340,    311,   5426,
# #             264,  19769,   1614, 151645,    198, 151644,  77091,    198]],
# #        device='cuda:0')
# # ['<|im_start|>', 'system', 'Ċ', 'ou', 'Ġare', 'ĠQ', 'wen', ',', 'Ġcreated', 'Ġby', 'ĠAlibaba', 'ĠCloud', '.', 'ĠYou', 'Ġare', 'Ġa', 'Ġhelpful', 'Ġassistant', '.', '<|im_end|>', 'Ċ', '<|im_start|>', 'user', 'Ċ', 'How', 'Ġto', 'Ġtrain', 'Ġa', 'ĠCNN', 'Ġmodel', '<|im_end|>', 'Ċ', '<|im_start|>', 'assistant', 'Ċ']
# text = tokenizer.apply_chat_template(
#     messages,
#     tokenize=False,
#     add_generation_prompt=True
# )
# model_inputs = tokenizer([prompt], return_tensors="pt").to(model.device)
# print(model_inputs["input_ids"])
# print(tokenizer.convert_ids_to_tokens(model_inputs["input_ids"][0]))
# cpu_time = time.time()
# generated_ids = model.generate(
#     **model_inputs,
#     max_new_tokens=512
# )
# generated_ids = [
#     output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
# ]

# response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

In [ ]:
print(response)